<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%91%D0%B5%D0%B7_%D0%92%D0%B7%D0%B0%D0%B8%D0%BC_%D0%9E%D0%B3%D1%80_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [1]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter, deque # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    print("Загружаем тренировочные данные заказов...")
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if 'created_date' in df.columns and df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    elif 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        # Если created_date вообще отсутствует, создаем её
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])
    print(f"Загружено заказов: {len(df):,}")
    return df

orders_df = load_orders()

Mounted at /content/drive
Загружаем тренировочные данные заказов...
Загружено заказов: 20,362,338


In [2]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [4]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:00<00:00,  1.21it/s]

Найдено уникальных тестовых пользователей: 470,347


In [5]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [6]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 20,362,338
Уникальных пользователей: 842,254
Уникальных товаров: 4,679,218
Период данных: 2025-01-01 - 2025-07-15

Распределение статусов заказов:
  delivered_orders: 10,420,894 (51.2%)
  canceled_orders: 8,420,631 (41.4%)
  proccesed_orders: 1,520,813 (7.5%)


In [7]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 51974017: 13,361 заказов
  Товар 187052809: 12,384 заказов
  Товар 207631139: 8,877 заказов
  Товар 143497612: 4,096 заказов
  Товар 119105606: 3,497 заказов
  Товар 247423473: 3,188 заказов
  Товар 77696741: 2,735 заказов
  Товар 175287070: 2,725 заказов
  Товар 285009143: 2,634 заказов
  Товар 201930716: 2,624 заказов


In [8]:
# Шаг 4. Разделение на тренировочную и тестовую выборки по времени
print("=== 📊 РАЗДЕЛЕНИЕ НА TRAIN/TEST ПО ВРЕМЕНИ ===")
train_end_date = pd.to_datetime('2025-07-08')  # Первая неделя: 2025-07-02 до 2025-07-08
test_start_date = pd.to_datetime('2025-07-09')  # Вторая неделя: 2025-07-09 до 2025-07-15

# Тренировочные данные (только доставленные заказы за первую неделю)
train_orders = orders_df[
    (orders_df['last_status'] == 'delivered_orders') &
    (orders_df['created_date'] >= pd.to_datetime('2025-07-02')) &
    (orders_df['created_date'] <= train_end_date)
].copy()

# Тестовые данные (заказы за вторую неделю - для оценки)
test_orders = orders_df[
    (orders_df['last_status'] == 'delivered_orders') &
    (orders_df['created_date'] >= test_start_date) &
    (orders_df['created_date'] <= pd.to_datetime('2025-07-15'))
].copy()

print(f"📅 Тренировочный период: 2025-07-02 - {train_end_date.date()}")
print(f"📅 Тестовый период: {test_start_date.date()} - 2025-07-15")
print(f"📦 Тренировочных заказов: {len(train_orders):,}")
print(f"📦 Тестовых заказов: {len(test_orders):,}")

=== 📊 РАЗДЕЛЕНИЕ НА TRAIN/TEST ПО ВРЕМЕНИ ===
📅 Тренировочный период: 2025-07-02 - 2025-07-08
📅 Тестовый период: 2025-07-09 - 2025-07-15
📦 Тренировочных заказов: 317,404
📦 Тестовых заказов: 167,241


In [9]:
# Шаг 5. Аналитическая "модель популярности" по тренировочным данным
def get_popular_items(train_orders, top_k=100):
    """Получить топ-K популярных товаров из тренировочной выборки"""
    print("\n=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===")
    item_counts = train_orders['item_id'].value_counts().head(top_k)
    popular_items = item_counts.index.tolist()

    print(f"Топ-{top_k} популярных товаров рассчитан")
    print("\n📋 Топ-10 самых популярных товаров:")
    for i, (item_id, count) in enumerate(item_counts.head(10).items(), 1):
        print(f"  {i:2d}. Товар {item_id:>12}: {count:>6,} заказов")

    return popular_items

popular_items = get_popular_items(train_orders, top_k=100)


=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===
Топ-100 популярных товаров рассчитан

📋 Топ-10 самых популярных товаров:
   1. Товар    175287070:    343 заказов
   2. Товар     51974017:    306 заказов
   3. Товар    166327353:    263 заказов
   4. Товар    187052809:    262 заказов
   5. Товар    247423473:    204 заказов
   6. Товар     11083343:    195 заказов
   7. Товар    334086992:    184 заказов
   8. Товар     63987378:    171 заказов
   9. Товар    206494927:    164 заказов
  10. Товар    201930716:    136 заказов


In [10]:
# Шаг 6. Получение тестовых пользователей (те, кто сделал заказы во вторую неделю)
test_users_during_period = test_orders['user_id'].unique().tolist()
print(f"\n=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
print(f"Найдено тестовых пользователей: {len(test_users_during_period):,}")


=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
Найдено тестовых пользователей: 96,159


In [11]:
# Преобразуем список тестовых пользователей в множество для быстрого поиска
test_users_set = set(test_users_during_period)
print(f"✅ Множество тестовых пользователей создано. Размер: {len(test_users_set):,}")

✅ Множество тестовых пользователей создано. Размер: 96,159


In [13]:
# Шаг 7. Вычисление истории покупок для тестовых пользователей (их покупки в первую неделю)
def build_user_history_first_week(orders_df, test_users, train_end_date):
    """Построить историю покупок тестовых пользователей до тестового периода"""
    print("\n=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (до теста) ===")

    # Все доставленные заказы тестовых пользователей до начала теста
    user_history = orders_df[
        (orders_df['last_status'] == 'delivered_orders') &
        (orders_df['user_id'].isin(test_users)) &
        (orders_df['created_date'] <= train_end_date)
    ].groupby('user_id')['item_id'].apply(set).to_dict()

    # Конвертируем в список для совместимости
    user_preferences = {uid: list(items) for uid, items in user_history.items()}

    print(f"История покупок построена для {len(user_preferences):,} пользователей")
    return user_preferences

user_preferences = build_user_history_first_week(orders_df, test_users_during_period, train_end_date)


=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (до теста) ===
История покупок построена для 94,024 пользователей


In [14]:
# Шаг 8. Генерация рекомендаций
def generate_recommendations(test_users, popular_items, user_preferences, top_k=100):
    """Генерация рекомендаций: популярные товары, исключая уже купленные"""
    print("\n=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===")
    recs = {}
    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        bought = set(user_preferences.get(uid, []))
        recs[uid] = [item for item in popular_items if item not in bought][:top_k]
    print(f"Рекомендации сгенерированы для {len(recs):,} пользователей")
    return recs

recommendations = generate_recommendations(test_users_during_period, popular_items, user_preferences)


=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===


Генерация рекомендаций: 100%|██████████| 96159/96159 [00:01<00:00, 82124.27it/s]

Рекомендации сгенерированы для 96,159 пользователей


In [15]:
# Шаг 10. Подготовка тестовых меток (что пользователи действительно купили во вторую неделю)
def prepare_ground_truth(test_orders):
    """Подготовить ground truth: что пользователи купили в тестовый период"""
    print("\n=== ✅ ПОДГОТОВКА GROUND TRUTH ===")
    ground_truth = test_orders.groupby('user_id')['item_id'].apply(set).to_dict()

    print(f"Ground truth подготовлен для {len(ground_truth):,} пользователей")

    # Показываем примеры для проверки
    if ground_truth:
        # Берем первых 3 пользователя для примера
        sample_users = sorted(list(ground_truth.keys()))[:3]  # Сортируем для воспроизводимости
        print("\n🔍 Примеры ground truth для проверки:")
        for i, user_id in enumerate(sample_users, 1):
            items = list(ground_truth[user_id])  # Все товары пользователя
            print(f"  Пользователь {user_id}: {items} ({len(ground_truth[user_id])} всего покупок)")

            # Дополнительная проверка - показываем те же данные из исходного тестового датафрейма
            user_test_data = test_orders[test_orders['user_id'] == user_id]
            test_items = user_test_data['item_id'].tolist()
            print(f"  Пользователь {user_id} (из теста): {test_items} ({len(test_items)} записей в тесте)")

    return ground_truth

ground_truth = prepare_ground_truth(test_orders)


=== ✅ ПОДГОТОВКА GROUND TRUTH ===
Ground truth подготовлен для 96,159 пользователей

🔍 Примеры ground truth для проверки:
  Пользователь 60: [18766015, 111169159] (2 всего покупок)
  Пользователь 60 (из теста): [111169159, 18766015] (2 записей в тесте)
  Пользователь 91: [27861200, 15428515, 27733496, 104094923] (4 всего покупок)
  Пользователь 91 (из теста): [15428515, 27861200, 27733496, 104094923] (4 записей в тесте)
  Пользователь 111: [179543784, 83982210] (2 всего покупок)
  Пользователь 111 (из теста): [179543784, 83982210] (2 записей в тесте)


In [16]:
# Шаг 10. Расчет NDCG@100
def ndcg_at_k(y_true, y_pred, k=100):
    """
    Рассчитать NDCG@k для одного пользователя
    y_true: множество реально купленных товаров
    y_pred: список рекомендованных товаров
    """
    if not y_true:
        return 0.0

    # Бинарная оценка: 1 если товар куплен, 0 если нет
    dcg = 0.0
    for i, item in enumerate(y_pred[:k]):
        if item in y_true:
            dcg += 1.0 / np.log2(i + 2)  # log2(1+pos) = log2(pos+2) т.к. индекс с 0

    # IDCG - идеальный DCG (все релевантные товары в начале)
    idcg = 0.0
    ideal_len = min(len(y_true), k)
    for i in range(ideal_len):
        idcg += 1.0 / np.log2(i + 2)

    if idcg == 0:
        return 0.0

    return dcg / idcg

def calculate_ndcg_batch(recommendations, ground_truth, k=100):
    """Рассчитать NDCG@k для всех пользователей"""
    print(f"\n=== 📊 РАСЧЕТ МЕТРИКИ NDCG@{k} ===")

    ndcg_scores = []
    users_evaluated = 0

    for uid in tqdm(recommendations.keys(), desc='Расчет NDCG'):
        if uid in ground_truth:
            y_pred = recommendations[uid]
            y_true = ground_truth[uid]

            ndcg = ndcg_at_k(y_true, y_pred, k)
            ndcg_scores.append(ndcg)
            users_evaluated += 1

    mean_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0.0
    print(f"Оценено пользователей: {users_evaluated:,}")
    print(f"NDCG@{k}: {mean_ndcg:.6f}")

    return mean_ndcg

# Расчет метрики
ndcg_score = calculate_ndcg_batch(recommendations, ground_truth, k=100)


=== 📊 РАСЧЕТ МЕТРИКИ NDCG@100 ===


Расчет NDCG: 100%|██████████| 96159/96159 [00:01<00:00, 75647.70it/s]

Оценено пользователей: 96,159
NDCG@100: 0.007415
